# NB_PRF_03 · Validar la propuesta de silver y gold contra los datos

**Paso 4 del framework. No escribe en silver ni en gold**: es un ensayo en seco (*dry run*) de lo que haría `NB_MDP_20_silver` con la metadata que propuso Claude.

Entrada: `config_json`, la salida de `meta.usp_GET_RUN_CONFIG` en la base de control **de desarrollo**, después de correr ahí el `alta_entidad.sql` propuesto. Es el mismo JSON que recibe `NB_MDP_00_orquestador`, así que se valida exactamente lo que va a correr.

Por cada entidad, sobre la última copia de bronze:

| Chequeo | Severidad si falla |
|---|---|
| Las columnas y expresiones de `meta.ENTITY_COLUMN` se pueden evaluar | BLOQUEANTE |
| Valores que se pierden al convertir a `TARGET_TYPE` (no nulos en bronze, nulos en silver) | ADVERTENCIA |
| Reglas `FAIL` que hoy cortarían la carga | BLOQUEANTE |
| Reglas `DROP` y `WARN`: cuántas filas afectan | ADVERTENCIA si pasa de `MAX_PCT_DROP` |
| `PRIMARY_KEY` con nulos | BLOQUEANTE |
| `PRIMARY_KEY` repetida en una misma copia (el `MERGE` se quedaría con una sola fila) | ADVERTENCIA |
| Columnas que el perfil marcó como PII y la metadata no (`IS_PII = 0`) | ADVERTENCIA |
| Columnas PII que aparecen en el `SQL_TEXT` de gold | ADVERTENCIA |
| `SQL_TEXT` de gold compila (`EXPLAIN`), si las tablas de silver ya existen | BLOQUEANTE |

Deja el resultado en `perfil.validacion` y en `Files/perfil/<origen>/<corrida_id>/validacion.md`, y lo devuelve con `exit`.

In [ ]:
# Parámetros
config_json = "{}"                # salida de meta.usp_GET_RUN_CONFIG (columna CONFIG_JSON)
ORIGEN = "erp"
CORRIDA_ID = ""                   # corrida de perfil para cruzar PII; vacío = la última de este origen
BRONZE_RUN_ID = ""                # vacío = el último _run_id de cada tabla de bronze
MAX_PCT_DROP = 1.0                # % de filas descartadas por reglas DROP que se considera alto
ESQUEMA_PERFIL = "perfil"
CARPETA_SALIDA = "/lakehouse/default/Files/perfil"

In [ ]:
import json, os, re
from datetime import datetime, timezone
from pyspark.sql import functions as F

cfg = json.loads(config_json)
if not CORRIDA_ID and spark.catalog.tableExists(f"{ESQUEMA_PERFIL}.corrida"):
    ult = (spark.table(f"{ESQUEMA_PERFIL}.corrida").where(F.col("origen") == ORIGEN)
                .orderBy(F.desc("inicio")).first())
    CORRIDA_ID = ult["corrida_id"] if ult else ""

pii_perfil = set()   # (tabla bronze, columna bronze) que el perfil marcó como PII
if CORRIDA_ID:
    pii_perfil = {(r["tabla"], r["columna"]) for r in spark.table(f"{ESQUEMA_PERFIL}.columna")
                  .where((F.col("corrida_id") == CORRIDA_ID) & F.col("pii")).collect()}

hallazgos = []


def hallazgo(objeto, chequeo, severidad, detalle, filas=None):
    hallazgos.append({"objeto": objeto, "chequeo": chequeo, "severidad": severidad,
                      "detalle": detalle, "filas": filas})


def leer_bronze(tabla):
    df = spark.table(tabla)
    if "_run_id" in df.columns:
        run = BRONZE_RUN_ID or (df.groupBy("_run_id").agg(F.max("_ingested_at").alias("m"))
                                  .orderBy(F.desc("m")).first()["_run_id"])
        df = df.where(F.col("_run_id") == run)
    return df

## 1. Entidades: columnas, tipos, reglas y clave

In [ ]:
pii_silver = {}   # silver_table -> columnas PII (según metadata o perfil)

for e in cfg.get("entities") or []:
    obj = e["entity_code"]
    if not spark.catalog.tableExists(e["bronze_table"]):
        hallazgo(obj, "bronze", "BLOQUEANTE", f"No existe {e['bronze_table']}: falta la carga exploratoria")
        continue
    bronze = leer_bronze(e["bronze_table"]).persist()
    filas = bronze.count()
    cols = e.get("columns") or []

    # 1.1 Expresiones y tipos: misma lógica que aplicar_columnas() de NB_MDP_20
    try:
        if cols:
            exprs, perdidas = [], []
            for c in cols:
                expr = F.expr(c.get("transform_expr") or f"`{c['source_column']}`")
                destino = expr.cast(c["target_type"])
                exprs.append(destino.alias(c["target_column"]))
                origen = F.col(f"`{c['source_column']}`")
                lleno = origen.isNotNull() & (F.trim(origen.cast("string")) != "")
                perdidas.append(F.sum((lleno & destino.isNull()).cast("int")).alias(c["target_column"]))
            for k, n in bronze.agg(*perdidas).first().asDict().items():
                if n:
                    hallazgo(obj, "tipo", "ADVERTENCIA",
                             f"{k}: {n} valores no nulos en bronze quedan NULL al convertir a "
                             f"{next(c['target_type'] for c in cols if c['target_column'] == k)}", n)
            silver = bronze.select(*exprs)
        else:
            silver = bronze.drop(*[c for c in bronze.columns if c.startswith("_")])
    except Exception as ex:
        hallazgo(obj, "columnas", "BLOQUEANTE", f"{type(ex).__name__}: {str(ex)[:500]}")
        bronze.unpersist()
        continue

    # 1.2 PII: el perfil y la metadata tienen que coincidir
    marcadas = {c["source_column"] for c in cols if c.get("is_pii")}
    for c in cols:
        if (e["bronze_table"], c["source_column"]) in pii_perfil and c["source_column"] not in marcadas:
            hallazgo(obj, "pii", "ADVERTENCIA",
                     f"{c['target_column']}: el perfil la marcó como PII y tiene IS_PII = 0. Confirmalo")
    pii_silver[e["silver_table"]] = {c["target_column"] for c in cols if c.get("is_pii")}

    # 1.3 Reglas de calidad, sin cortar en la primera
    for r in e.get("dq_rules") or []:
        try:
            n = silver.where(~F.coalesce(F.expr(r["rule_expr"]), F.lit(False))).count()
        except Exception as ex:
            hallazgo(obj, f"regla {r['rule_name']}", "BLOQUEANTE", f"No compila: {str(ex)[:300]}")
            continue
        if not n:
            continue
        pct = round(100 * n / filas, 2) if filas else 0
        if r["action"] == "FAIL":
            hallazgo(obj, f"regla {r['rule_name']}", "BLOQUEANTE",
                     f"FAIL: {n} filas ({pct}%) no cumplen {r['rule_expr']}: la carga de silver fallaría", n)
        elif r["action"] == "DROP":
            sev = "ADVERTENCIA" if pct > MAX_PCT_DROP else "INFO"
            hallazgo(obj, f"regla {r['rule_name']}", sev,
                     f"DROP: {n} filas ({pct}%) irían a dq_rechazos", n)
        else:
            hallazgo(obj, f"regla {r['rule_name']}", "INFO", f"WARN: {n} filas ({pct}%)", n)

    # 1.4 Clave
    pk = [k.strip() for k in (e.get("primary_key") or "").split(",") if k.strip()]
    if pk:
        faltan = [k for k in pk if k not in silver.columns]
        if faltan:
            hallazgo(obj, "clave", "BLOQUEANTE", f"PRIMARY_KEY {faltan} no está entre las columnas de silver")
        else:
            nulas = silver.where(" OR ".join(f"`{k}` IS NULL" for k in pk)).count()
            if nulas:
                hallazgo(obj, "clave", "BLOQUEANTE", f"{nulas} filas con la clave {pk} nula", nulas)
            rep = silver.groupBy(*pk).count().where("count > 1")
            claves_rep = rep.count()
            if claves_rep:
                filas_rep = rep.agg(F.sum("count")).first()[0]
                hallazgo(obj, "clave", "ADVERTENCIA",
                         f"{claves_rep} claves repetidas ({filas_rep} filas) en una sola copia: "
                         f"el MERGE se queda con una fila por clave. ¿La clave {pk} es la correcta?", filas_rep)
    elif e.get("silver_mode") == "MERGE":
        hallazgo(obj, "clave", "BLOQUEANTE", "SILVER_MODE = MERGE sin PRIMARY_KEY")

    hallazgo(obj, "resumen", "INFO", f"{filas} filas en la copia de bronze, {len(silver.columns)} columnas en silver")
    bronze.unpersist()

## 2. Gold: PII y compilación

In [ ]:
for g in cfg.get("gold_objects") or []:
    obj, sql = g["gold_code"], g["sql_text"]
    for tabla, pii in pii_silver.items():
        nombre = tabla.split(".")[-1]
        if re.search(rf"\b{re.escape(nombre)}\b", sql, re.I):
            usadas = [c for c in pii if re.search(rf"\b{re.escape(c)}\b", sql, re.I)]
            if usadas:
                hallazgo(obj, "pii", "ADVERTENCIA", f"usa columnas PII de {tabla}: {usadas}. ¿Hacen falta en gold?")
    referidas = set(re.findall(r"\b((?:\w+\.)?(?:silver|gold)\.\w+)\b", sql, re.I))
    faltan = [t for t in referidas if not spark.catalog.tableExists(t)]
    if faltan:
        hallazgo(obj, "compilar", "INFO", f"No se compila todavía: faltan {sorted(faltan)}. "
                                           "Se valida en la primera corrida del pipeline en desarrollo")
        continue
    try:
        texto = sql if g["load_mode"] != "MLV" else re.sub(r"(?is)^.*?\bAS\s+(SELECT|WITH)\b", r"\1", sql, count=1)
        spark.sql(f"EXPLAIN {texto}").collect()
        hallazgo(obj, "compilar", "INFO", "El SELECT compila")
    except Exception as ex:
        hallazgo(obj, "compilar", "BLOQUEANTE", f"{type(ex).__name__}: {str(ex)[:500]}")

## 3. Resultado

In [ ]:
orden = {"BLOQUEANTE": 0, "ADVERTENCIA": 1, "INFO": 2}
hallazgos.sort(key=lambda h: (orden[h["severidad"]], h["objeto"]))
ahora = datetime.now(timezone.utc)
VAL_ID = ahora.strftime("%Y%m%dT%H%M%SZ")
hay_bloqueantes = any(h["severidad"] == "BLOQUEANTE" for h in hallazgos)

if hallazgos:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ESQUEMA_PERFIL}")
    (spark.createDataFrame([(VAL_ID, ORIGEN, CORRIDA_ID, h["objeto"], h["chequeo"], h["severidad"], h["detalle"],
                             h["filas"], ahora) for h in hallazgos],
                           "validacion_id string, origen string, corrida_id string, objeto string, chequeo string, "
                           "severidad string, detalle string, filas bigint, validado_en timestamp")
          .write.format("delta").mode("append").saveAsTable(f"{ESQUEMA_PERFIL}.validacion"))

carpeta = os.path.join(CARPETA_SALIDA, re.sub(r"[^\w\-]+", "_", ORIGEN), CORRIDA_ID or "sin_perfil")
os.makedirs(carpeta, exist_ok=True)
md = [f"# Validación de la propuesta · {VAL_ID}", "",
      f"Resultado: **{'HAY BLOQUEANTES: no cargues esta metadata en producción' if hay_bloqueantes else 'sin bloqueantes'}**", "",
      "| Severidad | Objeto | Chequeo | Detalle |", "|---|---|---|---|"]
md += [f"| {h['severidad']} | `{h['objeto']}` | {h['chequeo']} | {h['detalle'].replace('|', '/')} |" for h in hallazgos]
open(os.path.join(carpeta, "validacion.md"), "w", encoding="utf-8").write("\n".join(md) + "\n")

display(spark.createDataFrame([(h["severidad"], h["objeto"], h["chequeo"], h["detalle"][:200]) for h in hallazgos]
                              or [("INFO", "-", "-", "sin entidades en config_json")],
                              "severidad string, objeto string, chequeo string, detalle string"))
resultado = {"validacion_id": VAL_ID, "hay_bloqueantes": hay_bloqueantes, "hallazgos": hallazgos}

In [ ]:
notebookutils.notebook.exit(json.dumps(resultado, ensure_ascii=False, default=str))